In [0]:
from pyspark.sql.functions import *

DataBase = "workspace.ecommerce"

details = spark.table(f"{DataBase}.silver_order_details")

In [0]:
monthly_revenue = (details.filter("order_status not in ('canceled', 'unavailable')")
                   .groupBy("purchase_month")
                   .agg(
                       sum("price").alias("revenue"),countDistinct("order_id").alias("orders")
                        )
                   .orderBy("purchase_month")
)

monthly_revenue.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DataBase}.gold_monthly_revenue")

monthly_revenue.show(20)

In [0]:
top_categories = (
    details.filter("order_status not in ('canceled', 'unavailable')")
           .groupBy("category")
           .agg(
               sum("price").alias("revenue"),
               count("order_id").alias("items_sold")
           )
           .orderBy(col("revenue").desc())
)

top_categories.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DataBase}.gold_top_categories")
top_categories.show(10, False)

In [0]:
delivery_by_state = (
    details.filter("delivered_ts is not null")
           .groupBy("state")
           .agg(
               count("order_id").alias("items_delivered"),
               round(avg("delivery_days"), 1).alias("avg_delivery_days"),
               round(avg("is_late") * 100, 2).alias("late_percent")
           )
           .orderBy(col("late_percent").desc())
)

delivery_by_state.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DataBase}.gold_delivery_by_state")
delivery_by_state.show(30)

In [0]:
orders = spark.table(f"{DataBase}.silver_orders")
customers = spark.table(f"{DataBase}.silver_customers")

customer_orders = (
    orders.join(customers, "customer_id", "inner")
          .groupBy("customer_unique_id")
          .agg(countDistinct("order_id").alias("order_count"))
          .withColumn("customer_type", expr("case when order_count > 1 then 'repeat' else 'one-time' end"))
)

repeat_summary = customer_orders.groupBy("customer_type").agg(count("customer_unique_id").alias("customers"))

repeat_summary.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DataBase}.gold_repeat_customers")
repeat_summary.show()

In [0]:
payments = spark.table(f"{DataBase}.silver_payments")

payment_summary = (
    payments.groupBy("payment_type")
            .agg(
                count("order_id").alias("payments"),
                sum("payment_value").alias("total_value")
            )
            .orderBy(col("total_value").desc())
)

payment_summary.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DataBase}.gold_payment_types")
payment_summary.show()